In [ ]:
#  !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

# Converting data to documents ( Example )

In [ ]:
from langchain_core.documents import Document

In [ ]:
# Example document from scratch
sample_doc = Document(
    page_content = "Hello World!",
    metadata = {"source" : "https://www.google.com"}
)

In [ ]:
sample_doc

In [ ]:
type(sample_doc)

In [ ]:
# Document from text

# -> Load the text file with the help of DataLoaders
from langchain_community.document_loaders.text import TextLoader
loader = TextLoader("data/Python.txt", encoding = "utf-8")

In [ ]:
# Load the document from the created loader object
document = loader.load()

In [ ]:
# Document from pdf
# import the loader
from langchain_community.document_loaders.pdf import PyPDFLoader
# We can also use PyMuPDFLoader, it will also work in the same way.

pdf_loader = PyPDFLoader("data/research1.pdf")
document_pdf = pdf_loader.load()

# 1. Ingestion Pipeline

## Data -> Documents

In [2]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

/var/folders/yq/61j601wx2psbghm1wpmrycxm0000gn/T/ipykernel_23457/220481367.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


In [3]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)
            
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs: ", num_docs)
    print("total pages: ", len(all_docs))

    return all_docs

In [4]:
all_pdf_documents = load_all_pdfs()

total pdfs:  2
total pages:  32


## Converting data into chunks

In [5]:
# !pip install langchain_text_splitters

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size = 500, chunk_overlap = 50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [7]:
chunks = split_doc(all_pdf_documents)

In [8]:
len(chunks)

320

## Embeddings

In [9]:
from sentence_transformers import SentenceTransformer

/opt/anaconda3/envs/datascience/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1125 12:30:39.383000 23457 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [10]:
# Creating a class - embedding manager to reuse the object to create embeddings whenever required
class EmbeddingManager:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):
        
        self.model_name = model_name
        print("loading model.....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions = ", self.model.get_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar = True)
        print("embedding shape : ", embeddings.shape)
        return embeddings

In [11]:
embedding_manager = EmbeddingManager()

loading model..... all-MiniLM-L6-v2


Loading weights: 100%|████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 12678.68it/s]


embedding dimensions =  384


## Vector Store

In [12]:
import hashlib
import chromadb
import uuid  # to create indexes for individual documents to uniquely identify them

In [13]:
class VectorStoreManager :
    def __init__(self, persist_directory = "data/vector_store", collection_name = "pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok = True)
        # create a client
        self.client = chromadb.PersistentClient(path = self.persist_directory)

        # create collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description" : "vector store collection for pdf embeddings."}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("number of documents doesn't match no. of embeddings")

        # store the ids, embeddings, documents(original text) and metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list  = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_key = str(doc.metadata.get('source', '')) + '_' + str(doc.metadata.get('page', 0)) + '_' + str(i)
            doc_id = hashlib.md5(doc_key.encode()).hexdigest()
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        self.collection.add(
            ids = ids,
            metadatas = all_metadata,
            documents = documents_content,
            embeddings = embeddings_list
        )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection", self.collection.count())

In [14]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 320


In [15]:
# data => documents => chunks => embeddings => store in vector store
texts =[doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 10/10 [00:01<00:00,  8.94it/s]

embedding shape :  (320, 384)
total documents added in vector store= 320
docs in collection 320


# 2. Retrieval Pipeline

In [16]:
# to retrieve the context from vector store after performing similarity search
# will return a list of results

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

In [18]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k = 5, score_threshold = 0.0):
        # query => embeddings
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k,
            include = ["documents", "metadatas", "embeddings"]
        )

        # cosine similarity
        retrieved_docs = []
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            embeddings = results["embeddings"][0]
            similarity_scores = cosine_similarity([query_embeddings], embeddings)[0]

            for i, (doc_id, metadata, document, score) in enumerate(zip(ids, metadatas, documents, similarity_scores)):
                similarity_score = float(score)

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                        "document": document,
                        "metadata": metadata,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
           print("no documents found") 

        return retrieved_docs

In [19]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [20]:
rag_retriever.retrieve("What is encoder decoder")

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 18.37it/s]

embedding shape :  (1, 384)
retrieved 5 documents


[{'id': 'c16cb094d8d5c82edcf19f0b019fa231',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7',
  'metadata': {'creator': 'PyPDF',
   'published': '2017',
   'creationdate': '',
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'page_label': '7',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'date': '2017',
   'book': 'Advances in Neural Information Processing Systems 30',
   'firstpage': '5998',
   'publisher': 'Curran Associates, Inc.',
   'doc_index': 293,
   'lastpage': '6008',
   'moddate': '2018-02-12T21:22:10-08:00',
   'source': 'data/pdfs/research1.pdf',
   'type': 'Conference Proceedings',
   'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)',
   'title': 'Attention is All you Need',
   'eventtype': 'Poster',
   'total_pages': 11

# Integrate with LLMs

## OpenAI - GPT

In [27]:
from dotenv import load_dotenv
load_dotenv()
API_KEY_OPENAI = os.getenv("OPENAI_API_KEY")

In [21]:
# !pip install langchain-openai

In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key = API_KEY_OPENAI,
    model = "gpt-5.4",
    temperature = 0.1,   # low for fact based things, high (>0.7) for creative work
    max_tokens = 1024
)

### Generate our Retrieval Augmented Output

In [23]:
def generate_output(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer
                Context : {context}
                Query : {query} """

    response = llm.invoke(prompt)  # expecting a string as prompt (OpenAI GPT expects string)
    return response.content

In [ ]:
answer = generate_output("what is RAG?", rag_retriever, llm)

In [ ]:
answer

## Groq LLM

In [28]:
API_KEY_GROQ = os.getenv("GROQ_API_KEY")

In [29]:
# !pip install langchain-groq

In [30]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = API_KEY_GROQ,
    model = "allam-2-7b",
    temperature = 0.1,
    max_tokens = 1024
)

In [31]:
def generate_output(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer
                Context : {context}
                Query : {query} """

    response = llm.invoke(prompt.format(context = context, query = query))  # expecting a list as prompt (Groq expects list)
    return response.content

In [32]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.64it/s]


embedding shape :  (1, 384)
retrieved 3 documents


In [33]:
answer

'RAG stands for Reinforcement Learning with Adversarial Games. In the context provided, RAG refers to the integration of Reinforcement Learning (RL) with Adversarial Games (AG) within Large Language Models (LLMs). The survey discusses the evolution of RAG methods, assessment methods, and potential future directions. The authors aim to provide a systematic synthesis of RAG to clarify its broader trajectory and focus on integrating RAG within LLMs. '

## Anthropic(Claude) LLM

In [34]:
# !pip install langchain-anthropic

In [35]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    anthropic_api_key = os.getenv("ANTHROPIC_API_KEY"),
    model = "claude-haiku-4-5-20251001",
    temperature = 0.1,
    max_tokens = 1024
)

In [36]:
def generate_output(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer
                Context : {context}
                Query : {query} """

    response = llm.invoke(prompt.format(context = context, query = query))  # expecting a list as prompt (Groq expects list) 
    # .invoke method is used to call the model.
    return response.content

In [ ]:
answer = generate_output("what is RAG?", rag_retriever, llm)